# CA19-9 Imputer Benchmark — KNN vs. MICE (Report Artifact)

In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, r"C:\FYP\src")
from utils.config import TABULAR_CLEAN_PATH, CLINICAL_IMPUTER_BENCHMARK_PATH, ensure_dirs, RANDOM_SEED
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score, recall_score
from xgboost import XGBClassifier

ensure_dirs()
pd.set_option("display.width", 120)
print("Imports OK")

Imports OK


## Load `tabular_clean.csv` and Reconstruct `feature_matrix` / `METADATA` / `TARGETS`

In [2]:
FEATURES = ["creatinine", "LYVE1", "REG1B", "TFF1", "plasma_CA19_9", "age", "sex"]
METADATA_COLS = ["sample_id", "patient_cohort", "sample_origin", "stage", "diagnosis"]
TARGET_COLS = ["dx", "target_binary"]

tabular_clean_df = pd.read_csv(TABULAR_CLEAN_PATH)
feature_matrix = tabular_clean_df[FEATURES].copy()
METADATA = tabular_clean_df[METADATA_COLS].copy()
TARGETS = tabular_clean_df[TARGET_COLS].copy()

print(f"feature_matrix: {feature_matrix.shape}   METADATA: {METADATA.shape}   TARGETS: {TARGETS.shape}")
print(f"plasma_CA19_9 missing: {feature_matrix['plasma_CA19_9'].isna().sum()}")

feature_matrix: (590, 7)   METADATA: (590, 5)   TARGETS: (590, 2)
plasma_CA19_9 missing: 240


## Section 7 — Two `plasma_CA19_9` Imputers, Built to Compare

In [3]:
class KNN_CA19_9Imputer:
    """Fold-safe KNN imputer for plasma_CA19_9. Fit on a training fold only,
    then apply to any fold (train or held-out) via transform().
    Never reads diagnosis / dx / target_binary.
    """

    PREDICTORS = ["creatinine", "LYVE1", "REG1B", "TFF1", "age"]
    TARGET = "plasma_CA19_9"

    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors
        self.scaler = StandardScaler()
        self.imputer = KNNImputer(n_neighbors=n_neighbors)

    def fit(self, train_df):
        observed = train_df[train_df[self.TARGET].notna()]
        self.scaler.fit(observed[self.PREDICTORS])
        scaled = self.scaler.transform(observed[self.PREDICTORS])
        fit_matrix = np.column_stack([scaled, observed[self.TARGET].values])
        self.imputer.fit(fit_matrix)
        return self

    def transform(self, target_df):
        out = target_df.copy()
        scaled = self.scaler.transform(out[self.PREDICTORS])
        matrix = np.column_stack([scaled, out[self.TARGET].values])
        imputed = self.imputer.transform(matrix)
        out[self.TARGET] = imputed[:, -1]
        return out

    def fit_transform(self, train_df, target_df=None):
        self.fit(train_df)
        return self.transform(train_df if target_df is None else target_df)


class MICE_CA19_9Imputer:
    """Fold-safe MICE (IterativeImputer + BayesianRidge) imputer for
    plasma_CA19_9. Same fit-on-training-fold-only, transform-with-already-
    fitted-imputer rules as KNN_CA19_9Imputer. No separate scaler -- fits
    directly on the training fold's predictors (always complete) + target
    (partially missing), which is exactly how IterativeImputer expects to
    be used. Never reads diagnosis / dx / target_binary.
    """

    PREDICTORS = ["creatinine", "LYVE1", "REG1B", "TFF1", "age"]
    TARGET = "plasma_CA19_9"

    def __init__(self, random_state=42):
        self.imputer = IterativeImputer(estimator=BayesianRidge(), random_state=random_state)

    def fit(self, train_df):
        self.imputer.fit(train_df[self.PREDICTORS + [self.TARGET]])
        return self

    def transform(self, target_df):
        out = target_df.copy()
        imputed = self.imputer.transform(out[self.PREDICTORS + [self.TARGET]])
        out[self.TARGET] = imputed[:, -1]
        return out

    def fit_transform(self, train_df, target_df=None):
        self.fit(train_df)
        return self.transform(train_df if target_df is None else target_df)


print("Imputer classes defined: KNN_CA19_9Imputer, MICE_CA19_9Imputer")

Imputer classes defined: KNN_CA19_9Imputer, MICE_CA19_9Imputer


**Smoke test — both imputers.**

In [4]:
rng = np.random.RandomState(0)
mask = rng.rand(len(feature_matrix)) < 0.8

for name, imputer_cls in [("KNN", KNN_CA19_9Imputer), ("MICE", MICE_CA19_9Imputer)]:
    smoke_imputer = imputer_cls()
    smoke_imputer.fit(feature_matrix[mask])

    filled_train = smoke_imputer.transform(feature_matrix[mask])
    filled_holdout = smoke_imputer.transform(feature_matrix[~mask])

    assert filled_train["plasma_CA19_9"].isna().sum() == 0
    assert filled_holdout["plasma_CA19_9"].isna().sum() == 0

    print(f"[{name}] train fold:   {mask.sum()} rows, "
          f"{feature_matrix[mask]['plasma_CA19_9'].isna().sum()} missing before, "
          f"{filled_train['plasma_CA19_9'].isna().sum()} missing after")
    print(f"[{name}] held-out fold: {(~mask).sum()} rows, "
          f"{feature_matrix[~mask]['plasma_CA19_9'].isna().sum()} missing before, "
          f"{filled_holdout['plasma_CA19_9'].isna().sum()} missing after")

print("\nSmoke test passed for both imputers: each fit on train-only, applied cleanly to train and held-out.")

[KNN] train fold:   473 rows, 199 missing before, 0 missing after
[KNN] held-out fold: 117 rows, 41 missing before, 0 missing after


[MICE] train fold:   473 rows, 199 missing before, 0 missing after
[MICE] held-out fold: 117 rows, 41 missing before, 0 missing after

Smoke test passed for both imputers: each fit on train-only, applied cleanly to train and held-out.


## Section 8 — A Shared, Imputer-Agnostic `run_fold`

In [5]:
EARLY_STAGES = {"I", "IA", "IB", "II", "IIA", "IIB"}  # resectability-based cut -- judgment call, flagged above


def run_fold(train_idx, test_idx, imputer_class, feature_matrix, targets, metadata, random_state=0):
    """Fit imputer_class() on train_idx only, transform both sides, then fit/score XGBoost.
    Never touches diagnosis/dx/target_binary inside the imputer.
    Returns a dict: auc, acc, pr_auc, early_recall."""
    train_fold = feature_matrix.loc[train_idx]
    test_fold = feature_matrix.loc[test_idx]
    y_train = targets.loc[train_idx, "target_binary"]
    y_test = targets.loc[test_idx, "target_binary"]

    imputer = imputer_class()
    imputer.fit(train_fold)
    train_imp = imputer.transform(train_fold)
    test_imp = imputer.transform(test_fold)

    assert train_imp["plasma_CA19_9"].isna().sum() == 0
    assert test_imp["plasma_CA19_9"].isna().sum() == 0

    model = XGBClassifier(n_estimators=100, max_depth=3, eval_metric="logloss", random_state=random_state)
    model.fit(train_imp[FEATURES], y_train)

    proba = model.predict_proba(test_imp[FEATURES])[:, 1]
    preds = model.predict(test_imp[FEATURES])
    preds_series = pd.Series(preds, index=test_idx)

    multi_class = y_test.nunique() > 1
    auc = roc_auc_score(y_test, proba) if multi_class else np.nan
    acc = accuracy_score(y_test, preds)
    pr_auc = average_precision_score(y_test, proba) if multi_class else np.nan

    is_early = metadata.loc[test_idx, "stage"].isin(EARLY_STAGES)
    early_idx = test_idx[is_early.values]
    early_recall = recall_score(y_test.loc[early_idx], preds_series.loc[early_idx]) if len(early_idx) > 0 else np.nan

    return {"auc": auc, "acc": acc, "pr_auc": pr_auc, "early_recall": early_recall}


print("run_fold() defined (imputer-agnostic, 4 metrics).")

run_fold() defined (imputer-agnostic, 4 metrics).


## Section 9 — Two-Imputer Comparison

In [6]:
IMPUTERS = {"KNN": KNN_CA19_9Imputer, "MICE": MICE_CA19_9Imputer}
comparison_rows = []  # accumulated across 9a/9b/9c: one row per (scheme, imputer)

# Splits computed ONCE, shared by both imputers below
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_SEED)
repeated_cv_splits = [
    (feature_matrix.index[train_pos], feature_matrix.index[test_pos])
    for train_pos, test_pos in rskf.split(feature_matrix, TARGETS["target_binary"])
]
print(f"Precomputed {len(repeated_cv_splits)} repeated-CV folds (shared across both imputers).\n")

for imputer_name, imputer_class in IMPUTERS.items():
    fold_results = [
        run_fold(train_idx, test_idx, imputer_class, feature_matrix, TARGETS, METADATA)
        for train_idx, test_idx in repeated_cv_splits
    ]
    fold_df = pd.DataFrame(fold_results)
    comparison_rows.append({
        "scheme": "Repeated 5x10 CV", "imputer": imputer_name, "n_folds": len(fold_df),
        "auc_mean": fold_df["auc"].mean(), "auc_std": fold_df["auc"].std(),
        "acc_mean": fold_df["acc"].mean(), "acc_std": fold_df["acc"].std(),
        "pr_auc_mean": fold_df["pr_auc"].mean(), "pr_auc_std": fold_df["pr_auc"].std(),
        "early_recall_mean": fold_df["early_recall"].mean(skipna=True),
        "early_recall_std": fold_df["early_recall"].std(skipna=True),
    })
    print(f"[{imputer_name}] Repeated Stratified 5-fold x 10-repeat CV -- {len(fold_df)} folds")
    print(f"  AUC:          {fold_df['auc'].mean():.3f} +/- {fold_df['auc'].std():.3f}")
    print(f"  Acc:          {fold_df['acc'].mean():.3f} +/- {fold_df['acc'].std():.3f}")
    print(f"  PR-AUC:       {fold_df['pr_auc'].mean():.3f} +/- {fold_df['pr_auc'].std():.3f}")
    print(f"  Early recall: {fold_df['early_recall'].mean(skipna=True):.3f} +/- "
          f"{fold_df['early_recall'].std(skipna=True):.3f}")
    print()

Precomputed 50 repeated-CV folds (shared across both imputers).



[KNN] Repeated Stratified 5-fold x 10-repeat CV -- 50 folds
  AUC:          0.891 +/- 0.027
  Acc:          0.810 +/- 0.032
  PR-AUC:       0.790 +/- 0.048
  Early recall: 0.641 +/- 0.097



[MICE] Repeated Stratified 5-fold x 10-repeat CV -- 50 folds
  AUC:          0.908 +/- 0.023
  Acc:          0.842 +/- 0.027
  PR-AUC:       0.841 +/- 0.041
  Early recall: 0.722 +/- 0.104



### 9b. Cohort-Out Split (train = Cohort1, test = Cohort2) — Both Imputers, Same Split

In [7]:
# Split computed ONCE (already deterministic -- fixed by patient_cohort), shared by both imputers
cohort_out_train_idx = METADATA.index[METADATA["patient_cohort"] == "Cohort1"]
cohort_out_test_idx = METADATA.index[METADATA["patient_cohort"] == "Cohort2"]

for imputer_name, imputer_class in IMPUTERS.items():
    result = run_fold(cohort_out_train_idx, cohort_out_test_idx, imputer_class, feature_matrix, TARGETS, METADATA)
    comparison_rows.append({
        "scheme": "Cohort-out", "imputer": imputer_name, "n_folds": 1,
        "auc_mean": result["auc"], "auc_std": np.nan,
        "acc_mean": result["acc"], "acc_std": np.nan,
        "pr_auc_mean": result["pr_auc"], "pr_auc_std": np.nan,
        "early_recall_mean": result["early_recall"], "early_recall_std": np.nan,
    })
    print(f"[{imputer_name}] Cohort-out -- train=Cohort1 (n={len(cohort_out_train_idx)}), "
          f"test=Cohort2 (n={len(cohort_out_test_idx)})")
    print(f"  AUC: {result['auc']:.3f}  Acc: {result['acc']:.3f}  "
          f"PR-AUC: {result['pr_auc']:.3f}  Early recall: {result['early_recall']:.3f}")

[KNN] Cohort-out -- train=Cohort1 (n=332), test=Cohort2 (n=258)
  AUC: 0.908  Acc: 0.864  PR-AUC: 0.629  Early recall: 0.714
[MICE] Cohort-out -- train=Cohort1 (n=332), test=Cohort2 (n=258)
  AUC: 0.837  Acc: 0.868  PR-AUC: 0.629  Early recall: 0.679


### 9c. Leave-One-Site-Out (over `sample_origin`) — Both Imputers, Same Splits

In [8]:
# Splits computed ONCE (already deterministic -- fixed by sample_origin), shared by both imputers
site_splits = {
    site: (METADATA.index[METADATA["sample_origin"] != site], METADATA.index[METADATA["sample_origin"] == site])
    for site in sorted(METADATA["sample_origin"].unique())
}

site_detail_rows = []
for imputer_name, imputer_class in IMPUTERS.items():
    for site, (train_idx, test_idx) in site_splits.items():
        result = run_fold(train_idx, test_idx, imputer_class, feature_matrix, TARGETS, METADATA)
        site_detail_rows.append({"imputer": imputer_name, "site": site, "n_test": len(test_idx), **result})

    site_df = pd.DataFrame([r for r in site_detail_rows if r["imputer"] == imputer_name])
    comparison_rows.append({
        "scheme": "Leave-one-site-out", "imputer": imputer_name, "n_folds": len(site_df),
        "auc_mean": site_df["auc"].mean(skipna=True), "auc_std": site_df["auc"].std(skipna=True),
        "acc_mean": site_df["acc"].mean(skipna=True), "acc_std": site_df["acc"].std(skipna=True),
        "pr_auc_mean": site_df["pr_auc"].mean(skipna=True), "pr_auc_std": site_df["pr_auc"].std(skipna=True),
        "early_recall_mean": site_df["early_recall"].mean(skipna=True),
        "early_recall_std": site_df["early_recall"].std(skipna=True),
    })
    print(f"[{imputer_name}] Leave-one-site-out")
    for _, row in site_df.iterrows():
        print(f"  site={row['site']:6s} n={row['n_test']:3d}  AUC={row['auc']:.3f}  Acc={row['acc']:.3f}  "
              f"PR-AUC={row['pr_auc']:.3f}  Early recall={row['early_recall']:.3f}")
    print(f"  Mean AUC (excl. NaN): {site_df['auc'].mean(skipna=True):.3f}\n")

site_detail_df = pd.DataFrame(site_detail_rows)

[KNN] Leave-one-site-out
  site=BPTB   n=409  AUC=0.809  Acc=0.746  PR-AUC=0.526  Early recall=0.676
  site=ESP    n= 29  AUC=0.775  Acc=0.793  PR-AUC=0.935  Early recall=0.875
  site=LIV    n=132  AUC=0.829  Acc=0.667  PR-AUC=0.899  Early recall=0.561
  site=UCL    n= 20  AUC=nan  Acc=0.700  PR-AUC=nan  Early recall=nan
  Mean AUC (excl. NaN): 0.805



[MICE] Leave-one-site-out
  site=BPTB   n=409  AUC=0.842  Acc=0.770  PR-AUC=0.629  Early recall=0.757
  site=ESP    n= 29  AUC=0.783  Acc=0.793  PR-AUC=0.943  Early recall=1.000
  site=LIV    n=132  AUC=0.834  Acc=0.727  PR-AUC=0.897  Early recall=0.667
  site=UCL    n= 20  AUC=nan  Acc=0.850  PR-AUC=nan  Early recall=nan
  Mean AUC (excl. NaN): 0.819



### 9d. Comparison Table

In [9]:
comparison_df = pd.DataFrame(comparison_rows)[
    ["scheme", "imputer", "n_folds", "auc_mean", "auc_std", "pr_auc_mean", "pr_auc_std",
     "early_recall_mean", "early_recall_std", "acc_mean", "acc_std"]
].round(4)

print("Scheme x Imputer comparison (all four metrics):")
display(comparison_df)

print("\nDeltas (MICE - KNN), by scheme:")
for metric in ["auc_mean", "pr_auc_mean", "early_recall_mean", "acc_mean"]:
    pivot = comparison_df.pivot(index="scheme", columns="imputer", values=metric)
    delta = (pivot["MICE"] - pivot["KNN"]).round(4)
    print(f"  {metric:20s}: " + ", ".join(f"{scheme}={v:+.4f}" for scheme, v in delta.items()))

Scheme x Imputer comparison (all four metrics):


,scheme,imputer,n_folds,auc_mean,auc_std,pr_auc_mean,pr_auc_std,early_recall_mean,early_recall_std,acc_mean,acc_std
0,Repeated 5x10 CV,KNN,50,0.8907,0.0271,0.7898,0.0476,0.6407,0.0970,0.8103,0.0319
1,Repeated 5x10 CV,MICE,50,0.9081,0.0229,0.8411,0.0411,0.7217,0.1037,0.8415,0.0268
2,Cohort-out,KNN,1,0.9084,NaN,0.6289,NaN,0.7143,NaN,0.8643,NaN
3,Cohort-out,MICE,1,0.8375,NaN,0.6288,NaN,0.6786,NaN,0.8682,NaN
4,Leave-one-site-out,KNN,4,0.8045,0.0271,0.7864,0.2265,0.7040,0.1587,0.7264,0.0550
5,Leave-one-site-out,MICE,4,0.8195,0.0322,0.8231,0.1696,0.8078,0.1724,0.7851,0.0511



Deltas (MICE - KNN), by scheme:
  auc_mean            : Cohort-out=-0.0709, Leave-one-site-out=+0.0150, Repeated 5x10 CV=+0.0174
  pr_auc_mean         : Cohort-out=-0.0001, Leave-one-site-out=+0.0367, Repeated 5x10 CV=+0.0513
  early_recall_mean   : Cohort-out=-0.0357, Leave-one-site-out=+0.1038, Repeated 5x10 CV=+0.0810
  acc_mean            : Cohort-out=+0.0039, Leave-one-site-out=+0.0587, Repeated 5x10 CV=+0.0312


## Write `results/clinical/imputer_benchmark.csv`

In [10]:
comparison_df.to_csv(CLINICAL_IMPUTER_BENCHMARK_PATH, index=False)
print(f"Wrote {CLINICAL_IMPUTER_BENCHMARK_PATH}")
print(f"Rows: {len(comparison_df)}")

Wrote C:\FYP\results\clinical\imputer_benchmark.csv
Rows: 6
